# 06 - Walk-Forward Validation

Notebook 05 showed that the regime-aware classifier can beat the baseline on one train/test split.

This notebook makes the evaluation more research-grade using walk-forward validation.

Input: `../data/processed/regime_features.csv`  
Output: `../data/processed/walk_forward_metrics.csv` and `../data/processed/walk_forward_predictions.csv`

## Why Walk-Forward Validation Matters

A single train/test split can be lucky or unlucky.

Walk-forward validation tests the model repeatedly through time:

```text
Train on past -> test on next period
Move forward
Train again -> test again
```

Important terms:

- **Look-ahead bias**: accidentally using future information.
- **Expanding window**: training set grows over time.
- **Rolling test window**: fixed future period used for testing.
- **Out-of-sample prediction**: prediction made on data the model has never seen.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Data

In [3]:
DATA_PATH = Path("../data/processed/regime_features.csv")
OUTPUT_DIR = Path("../data/processed")

regime_df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
regime_df = regime_df.sort_values("Date").reset_index(drop=True)

print(regime_df.shape)
print(regime_df["regime_label"].value_counts())
regime_df.head()

(623, 17)
regime_label
Bear      412
Bull      118
Crisis     93
Name: count, dtype: int64


,Date,ticker,Close,return_1d,volatility_30d,momentum_30d,ma_50_ratio,ma_200_ratio,drawdown,vix_close,target_return_1d,target_direction_1d,regime,regime_label,regime_0_prob,regime_1_prob,regime_2_prob
0,2010-04-27,^GSPC,1183.709961,-0.023382,0.007319,0.028857,0.020765,NaN,-0.027578,22.809999,0.006463,1,1,Bull,5.500702e-12,1.000000,1.553645e-11
1,2010-04-28,^GSPC,1191.359985,0.006463,0.007280,0.027513,0.025655,NaN,-0.021293,21.080000,0.012943,1,0,Bear,9.833277e-01,0.016623,4.970221e-05
2,2010-04-29,^GSPC,1206.780029,0.012943,0.007556,0.034788,0.037015,NaN,-0.008626,18.440001,-0.016648,0,0,Bear,9.994668e-01,0.000531,1.791272e-06
3,2010-04-30,^GSPC,1186.689941,-0.016648,0.008225,0.017893,0.018352,NaN,-0.025130,22.049999,0.013121,1,0,Bear,9.998941e-01,0.000065,4.078930e-05
4,2010-05-03,^GSPC,1202.260010,0.013121,0.008458,0.036520,0.030068,NaN,-0.012339,20.190001,-0.023838,0,0,Bear,9.986716e-01,0.000053,1.275341e-03


## 2. Prepare Features And Target

We use the same direction-classification setup as notebook 05.

In [4]:
feature_cols = [
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "drawdown",
    "vix_close",
    "regime_0_prob",
    "regime_1_prob",
    "regime_2_prob",
]

target_col = "target_direction_1d"
return_col = "target_return_1d"

data = regime_df.replace([np.inf, -np.inf], np.nan).copy()
data = data.dropna(subset=feature_cols + [target_col, return_col, "regime_label"]).reset_index(drop=True)
data[target_col] = data[target_col].astype(int)

print(data.shape)
print(data[["Date", "ticker", "Close", "regime_label", target_col]].head())
print(data[["Date", "ticker", "Close", "regime_label", target_col]].tail())

(623, 17)
        Date ticker        Close regime_label  target_direction_1d
0 2010-04-27  ^GSPC  1183.709961         Bull                    1
1 2010-04-28  ^GSPC  1191.359985         Bear                    1
2 2010-04-29  ^GSPC  1206.780029         Bear                    0
3 2010-04-30  ^GSPC  1186.689941         Bear                    1
4 2010-05-03  ^GSPC  1202.260010         Bear                    0
          Date ticker        Close regime_label  target_direction_1d
618 2025-11-19  ^GSPC  6642.160156         Bear                    0
619 2025-11-20  ^GSPC  6538.759766         Bear                    1
620 2025-11-21  ^GSPC  6602.990234         Bear                    1
621 2025-11-24  ^GSPC  6705.120117         Bear                    1
622 2025-11-25  ^GSPC  6765.879883         Bear                    1


## 3. Helper Functions

In [5]:
def build_classifier(max_depth=5, min_samples_leaf=10):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ))
    ])


def annualized_sharpe(returns, periods_per_year=252):
    returns = pd.Series(returns).dropna()
    if returns.std() == 0:
        return 0
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)


def max_drawdown(returns):
    returns = pd.Series(returns).fillna(0)
    equity_curve = (1 + returns).cumprod()
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1
    return drawdown.min()


def strategy_returns(actual_returns, up_prob, threshold=0.45):
    signal = (np.asarray(up_prob) >= threshold).astype(int)
    return signal * np.asarray(actual_returns)


def evaluate_fold(name, fold_id, y_true, up_prob, actual_returns, threshold=0.45):
    y_pred = (np.asarray(up_prob) >= threshold).astype(int)
    strat_returns = strategy_returns(actual_returns, up_prob, threshold)

    result = {
        "fold": fold_id,
        "model": name,
        "threshold": threshold,
        "rows": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "strategy_sharpe": annualized_sharpe(strat_returns),
        "strategy_max_drawdown": max_drawdown(strat_returns),
        "avg_daily_strategy_return": np.mean(strat_returns),
        "trade_rate": y_pred.mean(),
        "buy_hold_sharpe": annualized_sharpe(actual_returns),
        "buy_hold_max_drawdown": max_drawdown(actual_returns),
    }

    try:
        result["roc_auc"] = roc_auc_score(y_true, up_prob)
    except ValueError:
        result["roc_auc"] = np.nan

    return result

## 4. Define Walk-Forward Windows

We use row counts because this first dataset is already one asset and sorted by trading date.

Settings:

- Initial train size: 60% of data
- Test window: 125 rows, about 6 trading months
- Step size: 125 rows

In [6]:
initial_train_size = int(len(data) * 0.60)
test_window = 125
step_size = 125

windows = []
start = initial_train_size

while start + test_window <= len(data):
    train_start = 0
    train_end = start
    test_start = start
    test_end = start + test_window
    windows.append((train_start, train_end, test_start, test_end))
    start += step_size

print("Number of folds:", len(windows))
for i, window in enumerate(windows, start=1):
    train_start, train_end, test_start, test_end = window
    print(
        i,
        "Train:", data.iloc[train_start]["Date"].date(), "to", data.iloc[train_end - 1]["Date"].date(),
        "| Test:", data.iloc[test_start]["Date"].date(), "to", data.iloc[test_end - 1]["Date"].date(),
    )

Number of folds: 2
1 Train: 2010-04-27 to 2019-05-17 | Test: 2019-05-20 to 2021-05-17
2 Train: 2010-04-27 to 2021-05-17 | Test: 2021-05-18 to 2025-11-25


## 5. Run Walk-Forward Baseline And MoE

For each fold:

1. Train baseline on all training data.
2. Train experts on regime-specific training data.
3. Predict the next test window.
4. Store out-of-sample predictions.

In [ ]:
threshold = 0.45
min_rows_per_expert = 50

all_metrics = []
all_predictions = []

for fold_id, (train_start, train_end, test_start, test_end) in enumerate(windows, start=1):
    train_df = data.iloc[train_start:train_end].copy()
    test_df = data.iloc[test_start:test_end].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    baseline = build_classifier(max_depth=5, min_samples_leaf=10)
    baseline.fit(X_train, y_train)
    baseline_prob = baseline.predict_proba(X_test)[:, 1]

    expert_models = {}

    for regime_label in sorted(train_df["regime_label"].unique()):
        regime_train = train_df[train_df["regime_label"] == regime_label].copy()

        if len(regime_train) < min_rows_per_expert or regime_train[target_col].nunique() < 2:
            expert_models[regime_label] = baseline
            continue

        expert = build_classifier(max_depth=4, min_samples_leaf=5)
        expert.fit(regime_train[feature_cols], regime_train[target_col])
        expert_models[regime_label] = expert

    hard_prob = np.zeros(len(test_df))
    soft_prob = np.zeros(len(test_df))

    regime_to_state = (
        train_df[["regime", "regime_label"]]
        .drop_duplicates()
        .set_index("regime_label")["regime"]
        .to_dict()
    )

    test_reset = test_df.reset_index(drop=True)

    for i, row in test_reset.iterrows():
        regime_label = row["regime_label"]
        x_row = pd.DataFrame([row[feature_cols].values], columns=feature_cols)

        hard_expert = expert_models.get(regime_label, baseline)
        hard_prob[i] = hard_expert.predict_proba(x_row)[:, 1][0]

        weighted_prob = 0
        total_weight = 0

        for expert_label, expert in expert_models.items():
            if expert_label not in regime_to_state:
                continue

            state_number = regime_to_state[expert_label]
            prob_col = f"regime_{state_number}_prob"

            if prob_col not in row.index:
                continue

            weight = row[prob_col]
            expert_prob = expert.predict_proba(x_row)[:, 1][0]
            weighted_prob += weight * expert_prob
            total_weight += weight

        soft_prob[i] = weighted_prob / total_weight if total_weight > 0 else baseline_prob[i]

    actual_returns = test_df[return_col]

    all_metrics.append(evaluate_fold("Baseline", fold_id, y_test, baseline_prob, actual_returns, threshold))
    all_metrics.append(evaluate_fold("Hard MoE", fold_id, y_test, hard_prob, actual_returns, threshold))
    all_metrics.append(evaluate_fold("Soft MoE", fold_id, y_test, soft_prob, actual_returns, threshold))

    fold_predictions = test_df[["Date", "ticker", "Close", "regime_label", return_col, target_col]].copy()
    fold_predictions["fold"] = fold_id
    fold_predictions["baseline_up_prob"] = baseline_prob
    fold_predictions["hard_moe_up_prob"] = hard_prob
    fold_predictions["soft_moe_up_prob"] = soft_prob
    all_predictions.append(fold_predictions)

    print(f"Finished fold {fold_id}/{len(windows)}")

walk_metrics = pd.DataFrame(all_metrics)
walk_predictions = pd.concat(all_predictions, ignore_index=True)

walk_metrics

Finished fold 1/2


## 6. Aggregate Results

This is the main walk-forward result table.

In [ ]:
summary_metrics = (
    walk_metrics
    .groupby("model")
    .agg(
        folds=("fold", "count"),
        avg_accuracy=("accuracy", "mean"),
        avg_precision=("precision", "mean"),
        avg_recall=("recall", "mean"),
        avg_f1=("f1", "mean"),
        avg_roc_auc=("roc_auc", "mean"),
        avg_strategy_sharpe=("strategy_sharpe", "mean"),
        avg_strategy_max_drawdown=("strategy_max_drawdown", "mean"),
        avg_daily_strategy_return=("avg_daily_strategy_return", "mean"),
        avg_trade_rate=("trade_rate", "mean"),
        avg_buy_hold_sharpe=("buy_hold_sharpe", "mean"),
    )
    .sort_values("avg_strategy_sharpe", ascending=False)
)

summary_metrics

## 7. Save Results

In [ ]:
walk_metrics.to_csv(OUTPUT_DIR / "walk_forward_metrics.csv", index=False)
summary_metrics.to_csv(OUTPUT_DIR / "walk_forward_summary_metrics.csv")
walk_predictions.to_csv(OUTPUT_DIR / "walk_forward_predictions.csv", index=False)

print("Saved:", OUTPUT_DIR / "walk_forward_metrics.csv")
print("Saved:", OUTPUT_DIR / "walk_forward_summary_metrics.csv")
print("Saved:", OUTPUT_DIR / "walk_forward_predictions.csv")

## 8. Fold-by-Fold Accuracy

In [ ]:
plt.figure(figsize=(12, 5))

for model_name in walk_metrics["model"].unique():
    temp = walk_metrics[walk_metrics["model"] == model_name]
    plt.plot(temp["fold"], temp["accuracy"], marker="o", label=model_name)

plt.title("Walk-Forward Accuracy By Fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 9. Out-of-Sample Equity Curves

These curves use only predictions made in walk-forward test windows.

In [ ]:
def equity_curve(actual_returns, up_prob, threshold=0.45):
    returns = strategy_returns(actual_returns, up_prob, threshold)
    return (1 + pd.Series(returns)).cumprod()

walk_predictions = walk_predictions.sort_values("Date").reset_index(drop=True)

plt.figure(figsize=(14, 6))
plt.plot(
    walk_predictions["Date"],
    (1 + walk_predictions[return_col]).cumprod(),
    label="Buy & Hold",
    color="black",
)
plt.plot(
    walk_predictions["Date"],
    equity_curve(walk_predictions[return_col], walk_predictions["baseline_up_prob"], threshold),
    label="Baseline",
)
plt.plot(
    walk_predictions["Date"],
    equity_curve(walk_predictions[return_col], walk_predictions["hard_moe_up_prob"], threshold),
    label="Hard MoE",
)
plt.plot(
    walk_predictions["Date"],
    equity_curve(walk_predictions[return_col], walk_predictions["soft_moe_up_prob"], threshold),
    label="Soft MoE",
)

plt.title("Walk-Forward Out-of-Sample Equity Curves")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

## 10. Research Interpretation

Use this notebook to support a stronger project claim.

A good result statement looks like:

```text
Across walk-forward folds, the regime-aware MoE achieved higher average Sharpe and/or lower drawdown than the baseline, suggesting regime specialization improves risk-adjusted prediction.
```

If the MoE does not beat the baseline on every metric, that is still acceptable. In financial ML, the real argument is often about risk-adjusted behavior, not raw accuracy alone.

Next notebook options:

1. Add XGBoost/LightGBM experts.
2. Add all assets: S&P 500, NASDAQ, NIFTY.
3. Build final report notebook with charts and conclusions.